# Sesión 3 — Demo: Agente de Atención al Cliente con LangGraph

Notebook de demostración para la **Sesión 3** del curso de IA Generativa (ISDI MDA).

**El hilo conductor del curso:**
- Sesión 1: llamamos a un LLM directamente y exploramos prompting
- Sesión 2: indexamos reviews en ChromaDB y construimos un pipeline RAG
- Sesión 3: el vector store de la Sesión 2 se convierte en la **memoria de largo plazo** de un agente LangGraph

**El agente recibe un ticket de cliente y:**
1. Lo clasifica (categoría + urgencia)
2. Consulta el catálogo de reviews (RAG = memoria de largo plazo)
3. Redacta una respuesta fundamentada en datos reales
4. Decide si responder automáticamente o escalar a un humano

---

> **Backend `gemini`**: ejecutar en Google Colab con una API key en Secrets.  
> **Backend `ollama`**: ejecutar en local con Ollama corriendo (`ollama serve`).  
> El agente funciona igual con ambos backends.

In [1]:
%pip install -q \
    langgraph \
    langchain \
    langchain-google-genai \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    "datasets<3.0" \
    pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6

## 1. Configuración del modelo

In [19]:
# ── Configuración ────────────────────────────────────────────────────────────
BACKEND = "ollama"          # "gemini" | "ollama"
OLLAMA_MODEL = "qwen3.5:latest"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
# ─────────────────────────────────────────────────────────────────────────────

if BACKEND == "gemini":
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY
    )
    llm_json = llm  # Gemini usa tool calling para structured output; no necesita modo JSON aparte
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY
    )
    print("✓ Backend: Gemini 2.5 Flash + gemini-embedding-001")

elif BACKEND == "ollama":
    from langchain_ollama import ChatOllama, OllamaEmbeddings
    llm = ChatOllama(model=OLLAMA_MODEL)
    llm_json = ChatOllama(model=OLLAMA_MODEL, format="json")  # fuerza JSON a nivel de Ollama
    embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
    print(f"✓ Backend: Ollama — {OLLAMA_MODEL} + {OLLAMA_EMBEDDING_MODEL}")

else:
    raise ValueError(f"Backend desconocido: {BACKEND}")

✓ Backend: Ollama — qwen3.5:latest + nomic-embed-text-v2-moe


## 2. Memoria de largo plazo: el vector store de la Sesión 2

El agente necesita conocer el catálogo de productos para responder con datos reales.
Indexamos las reviews de Amazon Electronics — el mismo proceso que en la Sesión 2.

In [20]:
from datasets import load_dataset
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

print("Cargando reviews de Amazon Electronics...")
ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
df = pd.DataFrame(ds.take(300))[["title", "text", "rating", "parent_asin"]].dropna(subset=["text"])
df["rating"] = df["rating"].astype(int)

Cargando reviews de Amazon Electronics...


In [22]:
docs = [
    Document(
        page_content=f"{row['title']}\n\n{row['text']}",
        metadata={"rating": row["rating"], "parent_asin": row["parent_asin"]}
    )
    for _, row in df.iterrows()
]

print(f"Indexando {len(docs)} documentos en ChromaDB...")
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✓ Memoria de largo plazo lista: {vectorstore._collection.count()} vectores")

Indexando 300 documentos en ChromaDB...


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

## 3. Estado del agente

El estado es el diccionario tipado que fluye por todos los nodos del grafo.
Cada nodo recibe el estado completo y devuelve los campos que modifica.

In [ ]:
from typing import TypedDict, Literal

class TicketState(TypedDict):
    ticket: str           # mensaje original del cliente
    category: str         # devolucion | consulta_tecnica | queja | envio | otro
    urgency: str          # alta | media | baja
    summary: str          # resumen del ticket en 15 palabras
    rag_context: str      # reviews recuperadas del catálogo
    draft_response: str   # borrador generado por el LLM
    escalate: bool        # ¿escalar a un agente humano?
    final_response: str   # respuesta definitiva enviada al cliente

print("✓ Estado definido")

## 4. Nodos del agente

Cada nodo es una función pura: recibe el estado y devuelve un diccionario con los campos actualizados.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class ClasificacionTicket(BaseModel):
    category: Literal["devolucion", "consulta_tecnica", "queja", "envio", "otro"]
    urgency: Literal["alta", "media", "baja"]
    summary: str

clasificador = llm_json.with_structured_output(ClasificacionTicket)

def clasificar(state: TicketState) -> dict:
    """Nodo 1: clasifica el ticket en categoría, urgencia y resumen."""
    prompt = f"""Analiza el siguiente mensaje de un cliente.

Campos requeridos:
- category: una de [devolucion, consulta_tecnica, queja, envio, otro]
- urgency: una de [alta, media, baja]
- summary: resumen del problema en máximo 15 palabras

Mensaje del cliente:
{state['ticket']}"""

    datos = clasificador.invoke(prompt)
    print(f"  [clasificar] → categoría: {datos.category}, urgencia: {datos.urgency}")
    return {"category": datos.category, "urgency": datos.urgency, "summary": datos.summary}

print("✓ Nodo 'clasificar' definido (structured output)")

In [ ]:
def consultar_catalogo(state: TicketState) -> dict:
    """Nodo 2: recupera reviews relevantes del vector store (memoria de largo plazo)."""
    consulta = f"{state.get('category', '')} {state.get('summary', state['ticket'])}"
    docs = retriever.invoke(consulta)

    contexto = "\n\n---\n\n".join(
        f"[{d.metadata.get('rating', '?')}★] {d.page_content[:350]}"
        for d in docs
    )
    print(f"  [consultar_catalogo] → {len(docs)} reviews recuperadas")
    return {"rag_context": contexto}

print("✓ Nodo 'consultar_catalogo' definido")

In [ ]:
def redactar_respuesta(state: TicketState) -> dict:
    """Nodo 3: genera un borrador de respuesta usando el contexto recuperado."""
    prompt = f"""Eres el responsable de atención al cliente de una tienda de electrónica.
Tu tono es profesional, empático y directo.

Ticket del cliente:
{state['ticket']}

Categoría: {state.get('category', 'desconocida')} | Urgencia: {state.get('urgency', 'media')}

Información relevante del catálogo de productos:
{state.get('rag_context', 'Sin información disponible')}

Redacta una respuesta de no más de 3 frases que:
1. Reconozca el problema del cliente
2. Aporte información concreta del catálogo si es relevante
3. Indique el siguiente paso claro"""

    borrador = llm.invoke(prompt).content
    print(f"  [redactar_respuesta] → borrador generado ({len(borrador)} chars)")
    return {"draft_response": borrador}

print("✓ Nodo 'redactar_respuesta' definido")

In [ ]:
def evaluar_escalado(state: TicketState) -> dict:
    """Nodo 4: decide si el ticket debe escalarse a un agente humano."""
    categorias_criticas = {"devolucion", "queja"}
    escalar = (
        state.get("urgency") == "alta"
        and state.get("category") in categorias_criticas
    )
    print(f"  [evaluar_escalado] → escalar: {escalar}")
    return {"escalate": escalar}

def respuesta_automatica(state: TicketState) -> dict:
    """Nodo 5a: envía la respuesta automáticamente."""
    final = f"[RESPUESTA AUTOMÁTICA]\n\n{state['draft_response']}"
    print(f"  [respuesta_automatica] → enviada")
    return {"final_response": final}

def escalar_a_humano(state: TicketState) -> dict:
    """Nodo 5b: genera el contexto para que un agente humano retome el caso."""
    final = (
        f"[ESCALADO A AGENTE HUMANO]\n"
        f"Categoría: {state.get('category')} | Urgencia: {state.get('urgency')}\n"
        f"Resumen: {state.get('summary')}\n\n"
        f"Borrador sugerido para el agente:\n{state['draft_response']}"
    )
    print(f"  [escalar_a_humano] → ticket escalado")
    return {"final_response": final}

print("✓ Nodos 'evaluar_escalado', 'respuesta_automatica', 'escalar_a_humano' definidos")

## 5. Construcción del grafo LangGraph

In [ ]:
from langgraph.graph import StateGraph, END

def decidir_ruta(state: TicketState) -> str:
    """Función de enrutamiento condicional."""
    return "escalar" if state.get("escalate") else "enviar"

# Construir el grafo
builder = StateGraph(TicketState)

# Añadir nodos
builder.add_node("clasificar", clasificar)
builder.add_node("consultar_catalogo", consultar_catalogo)
builder.add_node("redactar_respuesta", redactar_respuesta)
builder.add_node("evaluar_escalado", evaluar_escalado)
builder.add_node("respuesta_automatica", respuesta_automatica)
builder.add_node("escalar_a_humano", escalar_a_humano)

# Definir el flujo
builder.set_entry_point("clasificar")
builder.add_edge("clasificar", "consultar_catalogo")
builder.add_edge("consultar_catalogo", "redactar_respuesta")
builder.add_edge("redactar_respuesta", "evaluar_escalado")
builder.add_conditional_edges(
    "evaluar_escalado",
    decidir_ruta,
    {"escalar": "escalar_a_humano", "enviar": "respuesta_automatica"}
)
builder.add_edge("escalar_a_humano", END)
builder.add_edge("respuesta_automatica", END)

# Compilar
agent = builder.compile()

print("✓ Agente compilado")

## 6. Visualización del grafo

In [ ]:
# Visualizar el grafo de estados
try:
    from IPython.display import display, Image
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: imprimir la representación Mermaid en texto
    print(agent.get_graph().draw_mermaid())

## 7. El agente en acción

Procesamos tickets reales y vemos el flujo paso a paso.

In [ ]:
def procesar_ticket(ticket: str) -> None:
    """Ejecuta el agente y muestra el resultado final."""
    print(f"\n{'='*60}")
    print(f"TICKET: {ticket}")
    print(f"{'='*60}")

    estado_inicial: TicketState = {
        "ticket": ticket,
        "category": "",
        "urgency": "",
        "summary": "",
        "rag_context": "",
        "draft_response": "",
        "escalate": False,
        "final_response": ""
    }

    resultado = agent.invoke(estado_inicial)

    print(f"\nRESULTADO FINAL:")
    print(resultado["final_response"])

In [ ]:
# Ticket 1: devolución urgente → debería escalarse
procesar_ticket(
    "Compré unos auriculares hace tres semanas y el sonido del oído derecho "
    "ha dejado de funcionar por completo. Quiero una devolución inmediata. "
    "Esto es completamente inaceptable."
)

In [ ]:
# Ticket 2: consulta técnica → respuesta automática
procesar_ticket(
    "Hola, ¿podéis decirme si el cable USB-C que vendéis en vuestra web "
    "es compatible con un MacBook Pro M3? Gracias."
)

In [ ]:
# Ticket 3: queja por retraso de envío → urgencia media, respuesta automática
procesar_ticket(
    "Mi pedido lleva 8 días de retraso sobre la fecha prometida. "
    "He intentado contactar con el servicio de atención al cliente dos veces "
    "sin respuesta. ¿Qué está pasando?"
)

## 8. Inspección del estado intermedio

Con LangGraph podemos ver el estado del agente después de cada nodo — fundamental para debugging y auditoría.

In [ ]:
ticket_inspeccion = (
    "El altavoz Bluetooth que compré hace un mes se apaga solo después de "
    "20 minutos de uso. He probado a resetearlo y sigue igual. Necesito solución."
)

estado_inicial: TicketState = {
    "ticket": ticket_inspeccion,
    "category": "", "urgency": "", "summary": "",
    "rag_context": "", "draft_response": "",
    "escalate": False, "final_response": ""
}

print("Traza de ejecución paso a paso:\n")
for step in agent.stream(estado_inicial):
    nodo = list(step.keys())[0]
    estado = step[nodo]
    campos_modificados = {k: v for k, v in estado.items() if v}
    print(f"[{nodo}] campos actualizados: {list(campos_modificados.keys())}")

---

## Resumen: lo que ha ocurrido

| Sesión | Pieza | Rol en el agente |
|--------|-------|------------------|
| Sesión 1 | Llamada directa al LLM | Base — el modelo es el motor de cada nodo |
| Sesión 2 | Pipeline RAG + ChromaDB | Memoria de largo plazo — el nodo `consultar_catalogo` |
| Sesión 3 | LangGraph | Orquestación — decide qué nodo ejecutar y en qué orden |

El agente no es una pieza nueva. Es la **composición** de todo lo anterior.